# Experiment A' (FELM): Claim-Type Asymmetry in ChatGPT Outputs

## Purpose
Experiment A (FRANK) uses pre-LLM summarisation models. A reviewer could ask whether the detection asymmetry transfers to LLM-generated content. FELM directly addresses this: it annotates **ChatGPT responses** across five domains that map onto CCL claim types.

## Data
**FELM** (Chen et al., NeurIPS 2023)
- 847 prompts → ChatGPT responses → **4,427 annotated segments**
- 5 domains: World Knowledge (wk), Math, Science, Reasoning, Writing/Recommendation
- Per-segment: binary factuality label (True=correct, False=error), error type

## CCL Mapping
| FELM Domain | CCL Category | Rationale |
|-------------|-------------|----------|
| wk (World Knowledge) | FACTUAL (■) | Verifiable entity/date/number claims |
| math | FACTUAL (■) | Verifiable computation |
| science | INTERPRETIVE (▲) | Causal/mechanistic reasoning |
| reasoning | INTERPRETIVE (▲) | Logical inference, framing |
| writing_rec | GAP (•) | Subjective/perspectival claims |

In [ ]:
import sys
sys.path.insert(0, '../..')

import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import numpy as np
from experiments.shared.plotting import setup_style, ccl_palette
setup_style()
palette = ccl_palette()

## 1. Load FELM Data

In [ ]:
from experiments.shared.data_acquisition import load_felm
from experiments.exp_a_felm.analysis import add_ccl_category

df = load_felm()
df = add_ccl_category(df)
print(f"{len(df)} segments from {df['record_index'].nunique()} prompts")
print(f"Overall error rate: {df['is_error'].mean():.1%}")
df.groupby(['domain', 'ccl_category']).agg(n=('is_error','count'), errors=('is_error','sum')).assign(rate=lambda x: x.errors/x.n)

## 2. Analysis 1 — Error Rate by CCL Category

In [ ]:
from experiments.exp_a_felm.analysis import (
    compute_error_rates, chi_square_error_rates, fisher_factual_vs_interpretive
)

by_domain, by_ccl = compute_error_rates(df)
print("By CCL category:")
display(by_ccl)

chi2 = chi_square_error_rates(by_ccl)
fisher = fisher_factual_vs_interpretive(by_ccl)
print(f"Chi-square: chi2={chi2['chi2']:.2f}, df={chi2['dof']}, p={chi2['p_value']:.4f}")
print(f"Fisher (FACTUAL vs INTERPRETIVE): OR={fisher['odds_ratio']:.2f}, p={fisher['p_value']:.4f}")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# By domain
bd = by_domain.sort_values(['ccl_category', 'domain'])
axes[0].bar(bd['domain'], bd['error_rate'],
            color=[palette.get(c, '#888') for c in bd['ccl_category']])
axes[0].set_title('Error Rate by Domain')
axes[0].set_ylabel('Error rate')
legend_elements = [mpatches.Patch(facecolor=palette.get(c,'#888'), label=c) for c in ['FACTUAL','INTERPRETIVE','GAP']]
axes[0].legend(handles=legend_elements)

# By CCL category
axes[1].bar(by_ccl['ccl_category'], by_ccl['error_rate'],
            color=[palette.get(c, '#888') for c in by_ccl['ccl_category']])
axes[1].set_title('Error Rate by CCL Category')
axes[1].set_ylabel('Error rate')

fig.suptitle('FELM: ChatGPT Error Rates by CCL Category', fontweight='bold')
fig.tight_layout()
plt.show()

## 3. Analysis 2 — Error Type Distribution

In [ ]:
from experiments.exp_a_felm.analysis import compute_error_type_distribution, chi_square_error_types

type_dist = compute_error_type_distribution(df)
display(type_dist)

chi2_types = chi_square_error_types(df)
if 'error' not in chi2_types:
    print(f"Chi-square (error types differ by CCL category): chi2={chi2_types['chi2']:.2f}, df={chi2_types['dof']}, p={chi2_types['p_value']:.4f}")
else:
    print(chi2_types['error'])

## 4. Analysis 3 — Segment Length as Detection Difficulty Proxy

In [ ]:
from experiments.exp_a_felm.analysis import compute_segment_lengths

seg = compute_segment_lengths(df)
display(seg['descriptive'])
if seg['p_value'] is not None:
    print(f"Factual error segments: {seg['factual_mean']:.1f} words")
    print(f"Interpretive error segments: {seg['interpretive_mean']:.1f} words")
    print(f"Mann-Whitney U={seg['mannwhitney_u']:.0f}, p={seg['p_value']:.4f}")

## 5. Integration with Experiment A (FRANK)

FRANK and FELM provide complementary evidence:

| Evidence | FRANK | FELM |
|----------|-------|------|
| Models | Pre-LLM (BART, PEGASUS...) | ChatGPT |
| Signal | Inter-annotator agreement | Error rate + type distribution |
| Finding | Factual errors: 2.4× more agreed-upon | (see above) |

**Combined interpretation**: The CCL's claim-type annotation system is motivated by two independent phenomena:
1. *Humans detect factual errors more reliably* (FRANK: OR=5.49)
2. *LLMs make systematically different types of errors by claim category* (FELM)

Together, they show that the category of a claim predicts both how likely an AI is to err and how likely a human is to catch it — the two failure modes the CCL is designed to address.